In [35]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/cleaned/supply_chain_clean.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (180519, 45)


In [36]:
date_columns = [
    "order_date_dateorders",
    "shipping_date_dateorders"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_columns].dtypes)

order_date_dateorders       datetime64[us]
shipping_date_dateorders    datetime64[us]
dtype: object


In [37]:
df["shipping_delay_days"] = (
    df["days_for_shipping_real"]
    - df["days_for_shipment_scheduled"]
)

In [65]:
df["delay_duration_days"] = (
    df["shipping_delay_days"].clip(lower=0)
)

print(df["delay_duration_days"].value_counts().sort_index())

delay_duration_days
0    77119
1    60647
2    28718
3     7052
4     6983
Name: count, dtype: int64


In [38]:
df["shipping_delay_days"].describe()

count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: shipping_delay_days, dtype: float64

In [39]:
df["order_to_shipping_days"] = (
    df["shipping_date_dateorders"]
    - df["order_date_dateorders"]
).dt.days

In [40]:
df["order_to_shipping_days"].describe()

count    180519.000000
mean          3.471856
std           1.670471
min           0.000000
25%           2.000000
50%           3.000000
75%           5.000000
max           6.000000
Name: order_to_shipping_days, dtype: float64

In [41]:
df["order_year"] = (
    df["order_date_dateorders"].dt.year
)

In [42]:
df["order_year"].value_counts().sort_index()

order_year
2015    62650
2016    62550
2017    53196
2018     2123
Name: count, dtype: int64

In [43]:
df["order_month"] = (
    df["order_date_dateorders"].dt.month
)

In [44]:
df["order_month_name"] = (
    df["order_date_dateorders"]
    .dt.month_name()
)

In [45]:
df["order_quarter"] = (
    df["order_date_dateorders"]
    .dt.quarter
)

In [46]:
df["order_day_of_week"] = (
    df["order_date_dateorders"]
    .dt.day_name()
)

In [47]:
df["order_week"] = (
    df["order_date_dateorders"]
    .dt.isocalendar()
    .week
    .astype(int)
)

In [48]:
def classify_shipping_performance(days):
    if days < 0:
        return "Faster than scheduled"
    elif days == 0:
        return "On schedule"
    else:
        return "Slower than scheduled"

df["shipping_performance"] = (
    df["shipping_delay_days"]
    .apply(classify_shipping_performance)
)

In [49]:
df["shipping_performance"].value_counts()

shipping_performance
Slower than scheduled    103400
Faster than scheduled     43366
On schedule               33753
Name: count, dtype: int64

In [50]:
df["value_per_item"] = (
    df["order_item_total"]
    / df["order_item_quantity"].replace(0, np.nan)
)

In [51]:
df["value_per_item"].describe()

count    180519.000000
mean        126.908898
std         126.299501
min           7.490000
25%          45.000000
50%          59.990002
75%         179.990005
max        1939.989990
Name: value_per_item, dtype: float64

In [52]:
new_features = [
    "shipping_delay_days",
    "order_to_shipping_days",
    "order_year",
    "order_month",
    "order_month_name",
    "order_quarter",
    "order_day_of_week",
    "order_week",
    "shipping_performance",
    "value_per_item"
]

df[new_features].head()

,shipping_delay_days,order_to_shipping_days,order_year,order_month,order_month_name,order_quarter,order_day_of_week,order_week,shipping_performance,value_per_item
0,-1,3,2018,1,January,1,Wednesday,5,Faster than scheduled,314.640015
1,1,5,2018,1,January,1,Saturday,2,Slower than scheduled,311.359985
2,0,4,2018,1,January,1,Saturday,2,On schedule,309.720001
3,-1,3,2018,1,January,1,Saturday,2,Faster than scheduled,304.809998
4,-2,2,2018,1,January,1,Saturday,2,Faster than scheduled,298.250000


In [53]:
print("Missing values in new features:")

print(
    df[new_features]
    .isnull()
    .sum()
)

Missing values in new features:
shipping_delay_days       0
order_to_shipping_days    0
order_year                0
order_month               0
order_month_name          0
order_quarter             0
order_day_of_week         0
order_week                0
shipping_performance      0
value_per_item            0
dtype: int64


In [54]:
print("Duplicate rows:", df.duplicated().sum())
print("Final shape:", df.shape)

Duplicate rows: 0
Final shape: (180519, 55)


In [55]:
df.to_csv(
    "../data/processed/supply_chain_feature_engineered.csv",
    index=False
)

print("Feature-engineered dataset saved.")
print("Final shape:", df.shape)

Feature-engineered dataset saved.
Final shape: (180519, 55)


In [56]:
df[[
    "shipping_delay_days",
    "order_to_shipping_days"
]].describe()

,shipping_delay_days,order_to_shipping_days
count,180519.000000,180519.000000
mean,0.565807,3.471856
std,1.490966,1.670471
min,-2.000000,0.000000
25%,0.000000,2.000000
50%,1.000000,3.000000
75%,1.000000,5.000000
max,4.000000,6.000000


In [57]:
# Order year
df["order_year"] = (
    df["order_date_dateorders"].dt.year
)

# Order month
df["order_month"] = (
    df["order_date_dateorders"].dt.month
)

# Month name
df["order_month_name"] = (
    df["order_date_dateorders"].dt.month_name()
)

# Quarter
df["order_quarter"] = (
    df["order_date_dateorders"].dt.quarter
)

# Day of week
df["order_day_of_week"] = (
    df["order_date_dateorders"].dt.day_name()
)

# Week number
df["order_week"] = (
    df["order_date_dateorders"]
    .dt.isocalendar()
    .week
    .astype(int)
)

print("Date features created successfully.")

Date features created successfully.


In [58]:
date_features = [
    "order_year",
    "order_month",
    "order_month_name",
    "order_quarter",
    "order_day_of_week",
    "order_week"
]

df[date_features].head(10)

,order_year,order_month,order_month_name,order_quarter,order_day_of_week,order_week
0,2018,1,January,1,Wednesday,5
1,2018,1,January,1,Saturday,2
2,2018,1,January,1,Saturday,2
3,2018,1,January,1,Saturday,2
4,2018,1,January,1,Saturday,2
5,2018,1,January,1,Saturday,2
6,2018,1,January,1,Saturday,2
7,2018,1,January,1,Saturday,2
8,2018,1,January,1,Saturday,2
9,2018,1,January,1,Saturday,2


In [59]:
def classify_shipping_performance(days):
    if days < 0:
        return "Faster than scheduled"
    elif days == 0:
        return "On schedule"
    else:
        return "Slower than scheduled"

df["shipping_performance"] = (
    df["shipping_delay_days"]
    .apply(classify_shipping_performance)
)

df["shipping_performance"].value_counts()

shipping_performance
Slower than scheduled    103400
Faster than scheduled     43366
On schedule               33753
Name: count, dtype: int64

In [60]:
df["value_per_item"] = (
    df["order_item_total"]
    / df["order_item_quantity"].replace(0, np.nan)
)

df["value_per_item"].describe()

count    180519.000000
mean        126.908898
std         126.299501
min           7.490000
25%          45.000000
50%          59.990002
75%         179.990005
max        1939.989990
Name: value_per_item, dtype: float64

In [61]:
new_features = [
    "shipping_delay_days",
    "order_to_shipping_days",
    "order_year",
    "order_month",
    "order_month_name",
    "order_quarter",
    "order_day_of_week",
    "order_week",
    "shipping_performance",
    "value_per_item"
]

print("Missing values:")
print(df[new_features].isnull().sum())

print("\nFinal shape:")
print(df.shape)

Missing values:
shipping_delay_days       0
order_to_shipping_days    0
order_year                0
order_month               0
order_month_name          0
order_quarter             0
order_day_of_week         0
order_week                0
shipping_performance      0
value_per_item            0
dtype: int64

Final shape:
(180519, 55)


In [62]:
df["delay_duration_days"] = (
    df["shipping_delay_days"].clip(lower=0)
)

In [63]:
df["delay_duration_days"].describe()

count    180519.000000
mean          0.926058
std           1.041748
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: delay_duration_days, dtype: float64

In [64]:
df["delay_duration_days"].value_counts().sort_index()

delay_duration_days
0    77119
1    60647
2    28718
3     7052
4     6983
Name: count, dtype: int64

In [66]:
df.to_csv(
    "../data/processed/supply_chain_feature_engineered.csv",
    index=False
)

print("Updated dataset saved successfully.")
print("Shape:", df.shape)

Updated dataset saved successfully.
Shape: (180519, 56)
